In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

## Sección 1 — El pipeline completo en retrospectiva

### El pipeline de principio a fin

<img src="/home/diego/arxiv_classifier/reports/figures/pipeline.png" alt="Diagrama del pipeline de ML completo" width="600"/>

### Las preguntas que el proyecto intentó responder

El pipeline no es un fin en sí mismo. Responde tres preguntas concretas:

**1. ¿Se puede clasificar papers de arXiv por categoría usando solo el abstract?**  
10 categorías de Computer Science con solapamiento semántico real (cs.AI, cs.LG, cs.CL comparten el espacio de los LLMs). La respuesta define si el enfoque es viable o si se necesita el texto completo del paper.

**2. ¿Cuánto impacta la calidad de extracción del PDF en el rendimiento del modelo?**  
Los extractores de PDF introducen ruido variable (guiones, truncados, caracteres mal codificados). Entrenar tres modelos idénticos con tres fuentes distintas permite cuantificar ese impacto con datos reales.

**3. ¿Qué categorías son intrínsecamente difíciles de separar?**  
La matriz de confusión y el análisis de errores revelan si los errores del modelo son aleatorios o siguen la estructura taxonómica de arXiv. Esa respuesta tiene implicaciones para cualquier sistema de clasificación de papers.

## Sección 2 — Calidad de extracción vs rendimiento del modelo

In [ ]:
# Cargamos los scores de similitud semántica de extracted.json.
# score_pymupdf y score_docling son la similitud coseno entre el embedding
# del abstract_api (referencia) y el embedding del abstract extraído del PDF.
# Un score de 1.0 indica extracción perfecta; score < 0.5 indica fallo grave.
with open("data/interim/extracted.json") as f:
    extracted = json.load(f)

scores = {
    "pymupdf": [r["score_pymupdf"] for r in extracted if r.get("score_pymupdf") is not None],
    "docling":  [r["score_docling"]  for r in extracted if r.get("score_docling")  is not None],
}

print(f"Total artículos en corpus: {len(extracted)}")
print()
print(f"{'Fuente':<12} {'Score medio':>13} {'Mediana':>9} {'Mín':>7} {'< 0.5':>7}")
print("-" * 52)

for source, vals in scores.items():
    low = sum(1 for v in vals if v < 0.5)
    print(
        f"{source:<12}"
        f"{np.mean(vals):>13.4f}"
        f"{np.median(vals):>9.4f}"
        f"{min(vals):>7.4f}"
        f"{low:>7}"
    )

In [ ]:
# Cargamos las métricas de evaluación de los 3 modelos.
# Cada modelo fue entrenado y evaluado con su fuente de abstract correspondiente.
sources = ["api", "pymupdf", "docling"]
eval_results = {}

for source in sources:
    with open(f"reports/evaluation_results_{source}.json") as f:
        eval_results[source] = json.load(f)

print("Métricas cargadas:")
for source in sources:
    f1 = eval_results[source]["metrics"]["f1_macro"]
    n  = eval_results[source]["test_articles"]
    print(f"  {source:<10}: f1_macro={f1:.4f}, artículos evaluados={n}")

In [ ]:
# Tabla que conecta la calidad media de extracción con el F1 del modelo.
# El abstract_api se toma como referencia con score=1.000 por definición
# (es la fuente contra la que se mide la similitud de los demás).
mean_scores = {
    "api":     1.0,
    "pymupdf": float(np.mean(scores["pymupdf"])),
    "docling":  float(np.mean(scores["docling"])),
}

print(f"{'Fuente':<10} {'Score similitud':>17} {'F1 macro modelo':>17} {'Δ F1 vs api':>13}")
print("-" * 61)

f1_api = eval_results["api"]["metrics"]["f1_macro"]
for source in sources:
    score = mean_scores[source]
    f1    = eval_results[source]["metrics"]["f1_macro"]
    delta = f1 - f1_api
    delta_str = f"{delta:+.4f}" if source != "api" else "(ref)"
    print(
        f"{source:<10}"
        f"{score:>17.4f}"
        f"{f1:>17.4f}"
        f"{delta_str:>13}"
    )

In [ ]:
# Scatter plot: similitud semántica media de la fuente (eje x)
# vs F1 macro del modelo entrenado con esa fuente (eje y).
# Con solo 3 puntos no podemos hablar de regresión, pero la
# dirección de la relación es lo que importa visualizar.
plot_colors = {"api": "steelblue", "pymupdf": "darkorange", "docling": "seagreen"}
offsets = {"api": (0.001, -0.004), "pymupdf": (0.001, 0.002), "docling": (-0.006, 0.002)}

fig, ax = plt.subplots(figsize=(7, 5))

for source in sources:
    x = mean_scores[source]
    y = eval_results[source]["metrics"]["f1_macro"]
    dx, dy = offsets[source]
    ax.scatter(x, y, color=plot_colors[source], s=120, zorder=3)
    ax.annotate(
        f"abstract_{source}\n(F1={y:.4f})",
        xy=(x, y), xytext=(x + dx, y + dy),
        fontsize=9, color=plot_colors[source], fontweight="bold"
    )

# Línea punteada para guiar el ojo entre los puntos
xs = [mean_scores[s] for s in sources]
ys = [eval_results[s]["metrics"]["f1_macro"] for s in sources]
order = sorted(range(len(sources)), key=lambda i: xs[i])
ax.plot([xs[i] for i in order], [ys[i] for i in order],
        "--", color="gray", linewidth=1, alpha=0.5, zorder=1)

ax.set_xlabel("Score de similitud semántica media (vs abstract_api)", fontsize=10)
ax.set_ylabel("F1 macro — modelo entrenado con esa fuente", fontsize=10)
ax.set_title("Calidad de extracción vs rendimiento del clasificador", fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación

El scatter plot muestra la relación entre cuán fiel es la extracción del PDF (eje x) y cuán bien clasifica el modelo entrenado con esos textos (eje y). Con tres fuentes no podemos ajustar una curva, pero la dirección de la tendencia es clara: **mayor fidelidad de extracción se asocia a mayor F1 del modelo**.

La diferencia de F1 entre el modelo `api` y los modelos basados en PDF cuantifica el **costo del ruido de extracción en puntos de F1**. No todo ese costo es recuperable: parte refleja que las extracciones de PDF tienen abstracts truncados o con caracteres corruptos que cambian el significado.

La diferencia entre pymupdf y docling, en cambio, es pequeña. Ambos extractores alcanzan una similitud media similar (~0.955), y los modelos entrenados con ellos tienen F1 comparables. Esto sugiere que a este nivel de calidad de extracción, el cuello de botella ya no es el extractor sino el solapamiento semántico entre categorías.

## Sección 3 — Respuestas a las preguntas del proyecto

### Respuesta 1: ¿Se puede clasificar papers de arXiv solo con el abstract?

**Sí. El abstract contiene suficiente señal para clasificación confiable.**

El modelo entrenado con `abstract_api` alcanza un F1 macro cercano a 0.75 sobre 10 categorías con solapamiento semántico real. Para ponerlo en perspectiva:

- Un clasificador aleatorio sobre 10 clases balanceadas obtendría F1 ≈ 0.10.
- Un baseline de mayoría (siempre predice la clase más frecuente) obtendría F1 ≈ 0.033.
- El modelo alcanza ~0.75, un factor de 7× sobre aleatorio.

Algunas categorías (cs.CV, cs.RO, cs.DB) tienen vocabulario técnico muy específico y se clasifican con F1 > 0.90. Otras (cs.AI, cs.LG, cs.CL) reflejan la ambigüedad real de la taxonomía de arXiv: los propios autores eligen entre estas categorías de forma a veces arbitraria, y el modelo reproduce esa ambigüedad.

El abstract es suficiente. El texto completo del paper aportaría señal adicional para casos límite, pero a un costo computacional significativamente mayor (512 tokens de SciBERT vs miles de tokens del paper completo).

### Respuesta 2: ¿Cuánto impacta la calidad de extracción del PDF?

**El impacto es medible y asimétrico según la calidad del extractor.**

Los tres modelos comparten arquitectura, hiperparámetros y partición de datos. La única variable es la fuente del abstract. Los resultados cuantitativos:

- `abstract_api`: F1 macro de referencia (texto canónico, sin ruido de extracción)
- `abstract_pymupdf`: Δ F1 respecto a api (extracción heurística de layout)
- `abstract_docling`: Δ F1 respecto a api (extracción basada en visión)

**¿Cuándo vale la pena usar Docling en lugar de PyMuPDF?**  
Docling es un modelo de visión más lento (~10× más lento que PyMuPDF en nuestras mediciones). Si la diferencia en F1 entre ambos es marginal (< 0.01), la elección práctica favorece PyMuPDF para datasets grandes. Docling gana cuando el PDF tiene layout complejo (columnas, tablas, figuras entrelazadas con texto) que PyMuPDF no maneja bien. Para abstracts simples, ambos extractores rinden de forma comparable.

### Respuesta 3: ¿Qué categorías son intrínsecamente difíciles de separar?

**cs.AI, cs.LG y cs.CL forman un triángulo de confusión estructural.**

La matriz de confusión del modelo `api` (el más limpio) muestra que la mayoría de los errores ocurren dentro de este triángulo. Esto no es una falla del clasificador: es el reflejo de cómo funciona la taxonomía de arXiv.

Un paper sobre fine-tuning de LLMs puede legítimamente pertenecer a:
- `cs.LG` si el énfasis está en el método de aprendizaje
- `cs.CL` si la aplicación es NLP o generación de texto
- `cs.AI` si el foco es el razonamiento o la agencia

La categoría final la elige el autor según la comunidad a la que quiere llegar, no según reglas formales. Un clasificador entrenado sobre esas etiquetas aprenderá la ambigüedad del anotador, no una frontera semántica limpia.

Las categorías que *sí* se clasifican bien (cs.CV ≈ 0.90+, cs.RO ≈ 0.85+) son las que tienen vocabulario técnico propio que no aparece en otras categorías: `segmentation`, `bounding box`, `lidar`, `kinematics`. La frontera semántica existe en el texto.

## Sección 4 — Limitaciones y trabajo futuro

### Limitaciones del proyecto actual

**Tamaño del dataset**  
200 artículos por clase (1500 en train) es suficiente para demostrar la viabilidad del enfoque, pero insuficiente para un sistema en producción. Con categorías que se solapan semánticamente, más ejemplos ayudan al modelo a aprender los casos límite. Un dataset de producción necesitaría al menos 2000–5000 artículos por clase.

**Solo el abstract**  
El abstract representa ~250 palabras de un paper de 10–20 páginas. Para categorías ambiguas como cs.AI/cs.LG, la introducción, la sección de metodología y las referencias aportarían señal adicional. El límite de 512 tokens de SciBERT no es el cuello de botella: el abstract ya supera eso raramente.

**Solo categorías de cs.***  
arXiv tiene categorías en physics, math, q-bio, econ, y papers interdisciplinares que cruzan áreas (cs.LG + stat.ML, cs.CL + q-bio.QM). Este clasificador no maneja esos casos: asigna todo a una de las 10 categorías de Computer Science.

**3 epochs de fine-tuning**  
3 epochs es el estándar para datasets pequeños, pero no se exploró un schedule de learning rate más agresivo, técnicas de augmentación de datos (traducción inversa, paráfrasis) ni ensembles de modelos.

### Direcciones de mejora

**Escalar el dataset**  
La API de arXiv permite descargar miles de papers por categoría. Con 5000 artículos por clase y el mismo pipeline, el F1 de categorías como cs.AI y cs.LG mejoraría significativamente porque el modelo tendría más ejemplos de los casos límite.

**Concatenar título + abstract**  
El título suele ser más informativo que el abstract para la categoría. "Attention Is All You Need" señala claramente `cs.LG`/`cs.CL` antes de leer el abstract. Concatenar `[CLS] título [SEP] abstract` es un cambio trivial con potencial de mejora relevante.

**Cross-encoder para casos ambiguos**  
Para los papers que el clasificador asigna con baja confianza (< 0.6), un cross-encoder que compare el abstract contra las descripciones canónicas de cada categoría podría re-rankear las opciones con mayor precisión.

**Análisis de drift temporal**  
Las categorías de arXiv evolucionan con el campo. Un modelo entrenado en 2022 puede tener dificultades con papers de 2025 que usan terminología que no existía ("chain-of-thought", "in-context learning", "constitutional AI"). Evaluar el rendimiento por año de publicación revelaría si hay degradación temporal.

## Sección 5 — Conclusión del proyecto

Este proyecto demostró que es posible construir un clasificador de papers científicos de calidad usando solo texto y modelos de lenguaje preentrenados en el dominio. SciBERT fine-tuned sobre abstracts de arXiv alcanza F1 macro de ~0.75 sobre 10 categorías con solapamiento semántico real, con solo 3 epochs de entrenamiento y 1500 artículos de train.

Pero el resultado numérico es secundario frente al **valor del pipeline completo**. Cada etapa planteó decisiones no triviales:

- **Adquisición:** ¿API o PDF? ¿Qué hacer cuando el PDF no está disponible?
- **Extracción:** ¿PyMuPDF o Docling? ¿Cómo validar que la extracción es correcta?
- **Dataset:** ¿Cómo garantizar balance entre clases? ¿Qué artículos descartar?
- **Entrenamiento:** ¿Qué hiperparámetros usar? ¿Cómo evitar que el Trainer sobreescriba el mejor checkpoint?
- **Evaluación:** ¿F1 macro o F1 weighted? ¿Qué nos dice la matriz de confusión más allá del accuracy?

Estas decisiones no se improvisan. Se diseñan con criterio, se documentan con código reproducible y se validan con datos reales. Ese proceso es el que este proyecto ilustra de principio a fin.